# 04 — Stroke spatial transcriptomics: GSE233814 (component of GSE233815)

**Study:** Zucha et al., PNAS 2024, permanent MCAO. Official SuperSeries: GSE233815, spatial SubSeries: GSE233814.

This notebook **discovers actual sample filenames through GEO SOFT metadata**, downloads published spatial transcriptomic 10x matrices and exports candidate-expression summaries **per tissue section and spatial spot**.

**Important scope:** tissue spots are mixtures of cells. Without the original `DetailedRegionAnnoShort` author's spatial labels, no spot is described as "core" or "periphery" based on inferred location. This first spatial notebook is a section-level expression audit only. A separate author-annotation transfer is required for core/periphery claims.

This study has severely constrained biological replication across time, so tissue spots are not treated as independent mice. There are **no spot-level inferential p-values**. Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE233814 and https://github.com/LabGenExp/Spatial_MCAO.


In [ ]:
!pip -q install "pandas>=2.0" "scipy>=1.12" "requests>=2.31" "anndata>=0.10"
import os, gzip, re, sys, hashlib
from pathlib import Path
import numpy as np,pandas as pd, scipy.sparse as sp
from scipy.io import mmread
import requests
ROOT=Path(os.environ.get("IMACS_ROOT","/content/imacs_stroke_pet_spatial" if Path("/content").exists() else "imacs_stroke_pet_spatial"))
RAW=ROOT/"raw"; OUT=ROOT/"results";RAW.mkdir(parents=True,exist_ok=True);OUT.mkdir(parents=True,exist_ok=True)
BASE="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE233nnn/GSE233814/soft/GSE233814_family.soft.gz"
CAT=pd.read_csv("https://raw.githubusercontent.com/Touf00/retinal-nvu-computational-analysis/imacs-stroke-pet-screen/IMACS_Stroke_PET/config/candidate_catalog.csv").drop_duplicates("gene")


## GEO metadata discovery and provenance (fail explicitly if no spatial count matrix can be found)

In [ ]:
def download(u,p):
    if p.exists() and p.stat().st_size>0:return p
    with requests.get(u,timeout=300,stream=True) as r:
        r.raise_for_status()
        with p.open("wb") as f:
            for data in r.iter_content(1<<20):
                if data:f.write(data)
    return p
def digest(p):
    h=hashlib.sha256()
    with p.open("rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):h.update(b)
    return h.hexdigest()

soft=download(BASE,RAW/"GSE233814_family.soft.gz")
samples=[];current=None
with gzip.open(soft,"rt",errors="replace") as f:
    for line in f:
        line=line.rstrip("\n")
        if line.startswith("^SAMPLE = "):
            if current:samples.append(current)
            current={"gsm":line.split("=",1)[1].strip(),"supplementary":[],"characteristics":[]}
        elif current is not None and line.startswith("!Sample_"):
            k,_,value=line[1:].partition(" = ")
            if k=="Sample_supplementary_file":current["supplementary"].append(value.strip())
            elif k=="Sample_characteristics_ch1":current["characteristics"].append(value.strip())
            else:current[k.replace("Sample_","")]=value.strip()
if current:samples.append(current)

manifest=[]
for s in samples:
    urls=s["supplementary"]
    matrix=[u for u in urls if "matrix.mtx" in u.lower()]
    feats=[u for u in urls if "features.tsv" in u.lower() or "genes.tsv" in u.lower()]
    barcs=[u for u in urls if "barcodes.tsv" in u.lower()]
    manifest.append({"gsm":s["gsm"],"title":s.get("title",""),"source":s.get("source_name_ch1",""),
     "characteristics":"; ".join(s["characteristics"]),"matrix_url":matrix[0] if matrix else "",
     "features_url":feats[0] if feats else "","barcodes_url":barcs[0] if barcs else "",
     "all_supplementary_files":"; ".join(urls)})
M=pd.DataFrame(manifest)
M.to_csv(OUT/"04_spatial_GEO_exact_sample_manifest.csv",index=False)
display(M[["gsm","title","source","characteristics","matrix_url"]])
assert len(M)>=1, "GEO GSE233814 has no parsed samples"
M=M[(M.matrix_url!="")&(M.features_url!="")&(M.barcodes_url!="")].copy()
assert len(M)>=1, "No complete matrix/features/barcodes triple—inspect GEO manifest"


## Per-section candidate expression

One sample at a time limits memory. We use each spatial capture spot as a measurement **only for descriptive summaries**. We preserve the distribution across spots and report the number of spots per section.

In [ ]:
provenance=[];summary=[];spot_values=[]
for r in M.itertuples(index=False):
    d=RAW/r.gsm;d.mkdir(exist_ok=True)
    paths={}
    for k in ("matrix","features","barcodes"):
        u=getattr(r,f"{k}_url")
        # The URL is copied exactly from GEO sample metadata, not guessed
        p=download(u,d/u.rsplit("/",1)[-1])
        paths[k]=p
        provenance.append({"gsm":r.gsm,"title":r.title,"file":p.name,"sha256":digest(p),"bytes":p.stat().st_size})
    with gzip.open(paths["matrix"],"rb") as f:
        X=mmread(f).T.tocsr().astype("float32")
    feats=pd.read_csv(paths["features"],sep="\t",header=None,compression="gzip")
    genes=feats.iloc[:,1].astype(str) if feats.shape[1]>1 else feats.iloc[:,0].astype(str)
    genes=pd.Index(genes)
    with gzip.open(paths["barcodes"],"rt") as f:barcodes=[line.strip() for line in f]
    assert X.shape==(len(barcodes),len(genes))
    lib=np.asarray(X.sum(axis=1)).ravel()
    total_spots=X.shape[0]
    for gene in CAT.gene:
        idx=np.flatnonzero(genes==gene)
        if not len(idx):
            summary.append({"gsm":r.gsm,"section":r.title,"gene":gene,"n_spots":total_spots,
                            "pct_spots_detected":np.nan,"mean_log1p_cpm":np.nan,"median_log1p_cpm":np.nan})
            continue
        counts=np.asarray(X[:,idx].sum(axis=1)).ravel()
        logcpm=np.log1p(counts/np.maximum(lib,1)*1e6)
        summary.append({"gsm":r.gsm,"section":r.title,"gene":gene,"n_spots":total_spots,
                    "pct_spots_detected":float((counts>0).mean()*100),
                    "mean_log1p_cpm":float(logcpm.mean()),"median_log1p_cpm":float(np.median(logcpm))})
        for bc,v in zip(barcodes,logcpm):
            spot_values.append({"gsm":r.gsm,"spot_id":bc,"gene":gene,"log1p_cpm":float(v)})
    print("Processed",r.gsm,r.title,"spots:",total_spots)

pd.DataFrame(provenance).to_csv(OUT/"04_spatial_file_checksums.csv",index=False)
S=pd.DataFrame(summary);S.to_csv(OUT/"04_spatial_candidate_expression_by_section.csv",index=False)
pd.DataFrame(spot_values).to_csv(OUT/"04_spatial_candidate_values_by_spot_NO_REGIONS.csv.gz",index=False,compression="gzip")
display(S.head(40))


## Conservative scientific interpretation

The crucial missing ingredient for actual "core vs penumbra" analysis is **the corresponding original author-labeled spot metadata** (e.g., `DetailedRegionAnnoShort` in their integrated Seurat object). We will not draw lesion-region boundaries by hand or infer cell-specific RNA from tissue-level spot values. The original lab's Seurat objects and scripts must be interrogated for those labels and spatial coordinates.

- No two sections are treated as independent animals without explicit donor identities.
- No spot-based p-values.
- No cell-type-specific target conclusions from mixed spatial capture spots.
- No "PET tracer works" claim from this evidence alone.


In [ ]:
import json
(OUT/"04_manifest.json").write_text(json.dumps({
"superseries":"GSE233815","spatial_subseries":"GSE233814",
"biological_inference":"DESCRIPTIVE SECTION LEVEL ONLY",
"region_labels_available":False,
"source_software":"Zucha et al. Spatial_MCAO GitHub",
"n_samples":int(M.shape[0])
},indent=2))
print("Saved",sorted(f.name for f in OUT.iterdir()))